
# Cryptography Lab Assignment 10

## MT19937 Stream Cipher and Predictable Token Attack

### Objective

To understand why a statistical Pseudorandom Number Generator (PRNG) is unsuitable for cryptographic applications by implementing a toy stream cipher, recovering a weak seed, and analyzing predictable password reset tokens.

### Software Requirements
- Python 3.12
- Jupyter Notebook
- Python random module
- Python secrets module
- Python time module



## 1. Introduction to MT19937

MT19937 is a widely used pseudorandom number generator (PRNG), also known as the Mersenne Twister.

It generates a sequence of numbers that appears random but is actually determined by an initial value called the seed.

### Important Features

1. It has a state size of 19937 bits.
2. It has a period of 2^19937 - 1.
3. It produces deterministic sequences.
4. The same seed generates the same sequence.
5. It is fast and suitable for simulations and statistical applications.

### Why MT19937 Is Not Cryptographically Secure

MT19937 is not designed for security-sensitive applications.

- Its output is deterministic.
- Its internal state can be reconstructed from sufficient output.
- Predictable seeds make its output easier to guess.
- It does not provide cryptographic protection against attackers.

Therefore, it should not be used for password reset tokens, encryption keys, or authentication tokens.



## 2. Part A - MT19937 Stream Cipher

A stream cipher encrypts data one byte at a time using a sequence of pseudorandom bytes called a keystream.

In this assignment, MT19937 generates one 8-bit keystream value for every plaintext byte.

### Working Principle

1. Initialize MT19937 using a secret seed.
2. Generate one 8-bit pseudorandom value.
3. XOR the generated value with the plaintext byte.
4. Repeat this process for every byte.
5. The resulting bytes form the ciphertext.

### Encryption Formula

Ciphertext = Plaintext XOR Keystream

### Decryption Formula

Plaintext = Ciphertext XOR Keystream

The same function is used for both encryption and decryption because XOR has the following property:

A XOR B XOR B = A

### Implementation

We used Python's random module to initialize the MT19937 generator and getrandbits(8) to generate one byte of keystream per input byte.



## 3. Known Plaintext Attack

A Known Plaintext Attack (KPA) is a cryptographic attack in which an attacker knows some part of the original plaintext and its corresponding ciphertext.

In this assignment, the plaintext contains 14 consecutive characters 'A'.

The attacker knows that the sequence of 14 A characters exists somewhere in the plaintext, but does not know its exact location.

### How Known Plaintext Helps

The encryption formula is:

C = P XOR K

Where:
- C = Ciphertext
- P = Plaintext
- K = Keystream

Therefore:

K = C XOR P

If the attacker knows the plaintext and ciphertext bytes, the corresponding keystream bytes can be calculated.

### Attack Procedure

1. Generate plaintext containing a random prefix of 5–20 printable characters.
2. Append 14 consecutive A characters.
3. Encrypt the plaintext using a randomly selected 16-bit seed.
4. Try every possible seed from 0 to 65535.
5. Decrypt the ciphertext using each candidate seed.
6. Search for 14 consecutive A characters.
7. If found, identify the matching seed and plaintext.
8. Verify the complete decrypted plaintext.

### Seed Search Space Calculation

The seed is 16 bits long.

Number of possible seeds:

2^16 = 65,536

Therefore, the attacker must check a maximum of 65,536 possible seeds.



## 4. Part B - Predictable Password Reset Tokens

A password reset token is a unique value generated by an application when a user requests a password reset.

In this assignment, a simulated application generates a 16-byte reset token using MT19937.

The generator is seeded with the current Unix timestamp in integer seconds.

### Unix Timestamp

A Unix timestamp represents the number of seconds elapsed since January 1, 1970 (UTC).

### Token Generation Procedure

1. Record the current Unix timestamp T.
2. Initialize MT19937 using T as the seed.
3. Generate 16 consecutive 8-bit outputs.
4. Combine these outputs to create a 16-byte token.
5. Display the timestamp and token in hexadecimal format.

### Token Prediction Attack

An attacker observes the generated token and knows that the token was generated within a specific time interval.

In this assignment, the search interval is:

T - 300 seconds to T + 300 seconds.

The attacker performs the following steps:

1. Test every timestamp in the given interval.
2. Use each timestamp as a candidate MT19937 seed.
3. Generate a 16-byte candidate token.
4. Compare the candidate token with the observed token.
5. If both tokens match, recover the timestamp.
6. Regenerate the token using the recovered timestamp.
7. Verify that both tokens are identical.

### Why Timestamp-Based Seeding Is Vulnerable

Unix timestamps are predictable because they represent time in seconds.

If an attacker knows approximately when a token was generated, the possible seed values are limited.

Instead of searching a huge random seed space, the attacker only needs to test timestamps within the expected interval.

This makes the token predictable and unsuitable for secure password reset applications.

### Security Recommendation

Secure applications should use cryptographically secure random number generators instead of MT19937 for generating password reset tokens.



## 5. Pseudocode - Part A: Seed Recovery

START

1. Generate a random printable prefix of length 5–20.
2. Append 14 consecutive A characters to the prefix.
3. Select a random 16-bit seed.
4. Initialize MT19937 using the selected seed.
5. Generate keystream bytes.
6. XOR plaintext with keystream to produce ciphertext.
7. FOR each candidate seed from 0 to 65535:
      a. Initialize MT19937 using candidate seed.
      b. Generate keystream bytes.
      c. Decrypt ciphertext using XOR.
      d. Search for 14 consecutive A characters.
      e. IF found:
            Store recovered seed and plaintext.
            STOP searching.
8. Verify the recovered seed.
9. Verify the complete plaintext.

STOP



## 6. Pseudocode - Part B: Token Prediction

START

1. Record current Unix timestamp T.
2. Initialize MT19937 using T as the seed.
3. Generate 16 consecutive 8-bit outputs.
4. Combine outputs to generate a 16-byte token.
5. Store the generated token.

6. Define search interval:
      Start = T - 300
      End = T + 300

7. FOR each candidate timestamp from Start to End:
      a. Initialize MT19937 using candidate timestamp.
      b. Generate a 16-byte candidate token.
      c. Compare candidate token with observed token.
      d. IF both tokens match:
            Store recovered timestamp.
            STOP searching.

8. Regenerate token using recovered timestamp.
9. Compare regenerated token with original token.
10. Display number of timestamps tested and verification result.

STOP



## 7. Conclusion

In this assignment, we implemented a toy stream cipher using the MT19937 pseudorandom number generator.

In Part A, we performed encryption and decryption using XOR and demonstrated how a known plaintext pattern can help recover a weak 16-bit seed through brute-force searching.

In Part B, we generated a 16-byte password reset token using a Unix timestamp as the seed. We demonstrated that an attacker can predict the token by testing timestamps within a limited search interval.

This assignment demonstrates that MT19937 is suitable for simulations but not for cryptographic applications because its deterministic output and predictable seeds can expose sensitive information.

Cryptographic applications should use secure random number generators for encryption keys and authentication tokens.


In [1]:
%run part_a.py

===== PART A: MT19937 STREAM CIPHER =====

1. Encryption and Decryption Test
Original Message: Hello Cryptography
Encrypted (Hex): 22de6ebdbced0fa8279e452af71d482f77a6
Decrypted Message: Hello Cryptography
Test Successful: True

2. Generated Plaintext and Ciphertext
Plaintext: y2y{`.p]G)oh_%U%L>lAAAAAAAAAAAAAA
Actual Seed: 17331
Ciphertext (Hex): 34d71d16e21b2bd2d4330c17440d28b636772953404b8f73d0d39e3cd26e81223a

3. Brute Force Seed Recovery
Searching all 65,536 possible seeds...
Recovered Seed: 17331
Recovered Plaintext: y2y{`.p]G)oh_%U%L>lAAAAAAAAAAAAAA

4. Verification
Seed Correct: True
Plaintext Correct: True
Known 14 A's Found: True


In [2]:
%run part_b.py

===== PART B: PREDICTABLE RESET TOKEN =====

1. Token Generation
Original Timestamp: 1793981539
Generated Token (Hex): 173b069d89c34f4bce4a5c8cb1af59bd

2. Attacker Search
Search Start: 1793981239
Search End: 1793981839
Search Interval: T-300 to T+300

3. Prediction Results
Total Timestamps Tested: 301
Recovered Timestamp: 1793981539

4. Verification
Timestamp Correct: True
Regenerated Token: 173b069d89c34f4bce4a5c8cb1af59bd
Token Match: True
